In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
###patch start###
from mpl_toolkits.mplot3d.axis3d import Axis
if not hasattr(Axis, "_get_coord_info_old"):
    def _get_coord_info_new(self, renderer):
        mins, maxs, centers, deltas, tc, highs = self._get_coord_info_old(renderer)
        mins += deltas / 4
        maxs -= deltas / 4
        return mins, maxs, centers, deltas, tc, highs
    Axis._get_coord_info_old = Axis._get_coord_info  
    Axis._get_coord_info = _get_coord_info_new
###patch end###

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for c, z in zip(['r', 'g', 'b', 'y'], [30, 20, 10, 0]):
    xs = np.arange(20)
    ys = np.random.rand(20)

    # You can provide either a single color or an array. To demonstrate this,
    # the first bar of each set will be colored cyan.
    cs = [c] * len(xs)
    cs[0] = 'c'
    ax.bar(xs, ys, zs=z, zdir='y', color=cs, alpha=0.8)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

plt.show()

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/InitializationDaily.ipynb"

In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import ArticlesAnalyzer
from nltk.sentiment.util import *

#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections
import matplotlib.pyplot as plt
# %matplotlib inline

import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import Tfidf_sentimentVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

In [ ]:
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl')

In [ ]:
f, ax = plt.subplots(nrows=4, ncols=1, figsize=(15, 12))

sns.lineplot(x=df['Date'], y=df['Articles'].fillna(0), ax=ax[0], color='darkorange', label = 'modified')
sns.lineplot(x=df['Date'], y=df['Articles'].fillna(np.inf), ax=ax[0], color='dodgerblue', label = 'original')
ax[0].set_title('Fill NaN with 0', fontsize=14)
ax[0].set_ylabel(ylabel='Volume C10 Petrignano', fontsize=14)

mean_drainage = df['Articles'].mean()
sns.lineplot(x=df['Date'], y=df['Articles'].fillna(mean_drainage), ax=ax[1], color='darkorange', label = 'modified')
sns.lineplot(x=df['Date'], y=df['Articles'].fillna(np.inf), ax=ax[1], color='dodgerblue', label = 'original')
ax[1].set_title(f'Fill NaN with Mean Value ({mean_drainage:.0f})', fontsize=14)
ax[1].set_ylabel(ylabel='Volume C10 Petrignano', fontsize=14)

sns.lineplot(x=df['Date'], y=df['Articles'].ffill(), ax=ax[2], color='darkorange', label = 'modified')
sns.lineplot(x=df['Date'], y=df['Articles'].fillna(np.inf), ax=ax[2], color='dodgerblue', label = 'original')
ax[2].set_title(f'FFill', fontsize=14)
ax[2].set_ylabel(ylabel='Volume C10 Petrignano', fontsize=14)

sns.lineplot(x=df['Date'], y=df['Articles'].interpolate(), ax=ax[3], color='darkorange', label = 'modified')
sns.lineplot(x=df['Date'], y=df['Articles'].fillna(np.inf), ax=ax[3], color='dodgerblue', label = 'original')
ax[3].set_title(f'Interpolate', fontsize=14)
ax[3].set_ylabel(ylabel='Volume C10 Petrignano', fontsize=14)

for i in range(4):
    ax[i].set_xlim([date(2020, 6, 1), date(2022, 10, 1)])
    
plt.tight_layout()
plt.show()

In [ ]:
df_sentiment

In [ ]:
f, ax = plt.subplots(nrows=4, ncols=1, figsize=(15, 12))

sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(0), ax=ax[0], color='darkorange', label = 'modified')
sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(np.inf), ax=ax[0], color='dodgerblue', label = 'original')
ax[0].set_title('Fill NaN with 0', fontsize=20)
ax[0].set_ylabel(ylabel='Sentiment', fontsize=14)

mean_Sentiment = df_sentiment['Sentiment'].mean()
sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(mean_Sentiment), ax=ax[1], color='darkorange', label = 'modified')
sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(np.inf), ax=ax[1], color='dodgerblue', label = 'original')
ax[1].set_title(f'Fill NaN with Mean Value ({mean_Sentiment:.0f})', fontsize=20)
ax[1].set_ylabel(ylabel='Sentiment', fontsize=14)

sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].ffill(), ax=ax[2], color='darkorange', label = 'modified')
sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(np.inf), ax=ax[2], color='dodgerblue', label = 'original')
ax[2].set_title(f'FFill', fontsize=20)
ax[2].set_ylabel(ylabel='Sentiment', fontsize=14)

sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].interpolate(), ax=ax[3], color='darkorange', label = 'modified')
sns.lineplot(x=df_sentiment['Date'], y=df_sentiment['Sentiment'].fillna(np.inf), ax=ax[3], color='dodgerblue', label = 'original')
ax[3].set_title(f'Interpolate', fontsize=20)
ax[3].set_ylabel(ylabel='Sentiment', fontsize=14)
# import time
# for i in range(4):
#     ax[i].set_xlim([date(2019, 9, 25), date(2021, 8, 13)])
    
if (SHOW_GRAPHS):
    plt.tight_layout()
    plt.show()